# TNC GeoTIFF Modeling Demo

This notebook is the Binder landing page for the project. It uses `Feature_Engineering.py` and `ML_Modeling.py` to turn GeoTIFF predictor layers into a modeling dataframe, create visualizations, and train regression models for a target raster.

The full workflow needs the project TIFF files in the expected folder. If those files are not present in Binder, the setup cell below will show which files are missing and skip the heavy modeling cells until the data is added.

## 1. Import libraries and project classes

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from Feature_Engineering import Feature_Engineering
from ML_Modeling import ML_Modeling

pd.set_option("display.max_columns", None)

## 2. Set input paths

The default example uses the Chimney Springs P-dry study area.

In [ ]:
data_dir = Path("ML_Modeling_Files/TIFF_Files_For_Model/Chimney_Springs_P-dry")

predictor_paths = [
    data_dir / "CC.tif",
    data_dir / "CH.tif",
    data_dir / "East.tif",
    data_dir / "North.tif",
    data_dir / "PAG.tif",
    data_dir / "SFI.tif",
    data_dir / "SVF.tif",
    data_dir / "WSI.tif",
]

target_path = data_dir / "PSWE_2020_SnowPALM_Map.tif"
target_variable = target_path.stem

all_paths = predictor_paths + [target_path]
missing_files = [path for path in all_paths if not path.exists()]

if missing_files:
    print("Missing GeoTIFF files:")
    for path in missing_files:
        print(f"- {path}")
else:
    print("All GeoTIFF files found. Ready to run the workflow.")

## 3. Build the modeling dataframe

This converts the raster layers into one dataframe, with one row per valid pixel.

In [ ]:
feature_engineering = Feature_Engineering()

if missing_files:
    print("Add the missing TIFF files, then rerun this notebook cell.")
else:
    processed_df = feature_engineering.preprocess_data(
        [str(path) for path in predictor_paths],
        str(target_path),
    )
    print(processed_df.shape)
    display(processed_df.head())

## 4. Create feature visualizations

This creates a correlation matrix, pair plot, and predictor-vs-target scatter plots as PNG files in the notebook folder.

In [ ]:
if missing_files:
    print("Skipping visualizations because the TIFF files are missing.")
else:
    feature_engineering.create_visualizations_include_scatter(
        [str(path) for path in predictor_paths],
        str(target_path),
        target_variable,
    )
    print("Visualization files created.")

## 5. Train and evaluate models

This step can take a while because it runs randomized hyperparameter searches for multiple tree-based models.

In [ ]:
ml_modeling = ML_Modeling()

if missing_files:
    print("Skipping model training because the TIFF files are missing.")
else:
    ml_modeling.create_ML_Model(
        [str(path) for path in predictor_paths],
        str(target_path),
        target_variable,
    )
    print("Model training complete.")

## 6. Read saved metrics

After training, the model metrics are saved as `.joblib` files.

In [ ]:
metric_files = [
    "et_metrics_train.joblib",
    "et_metrics_test.joblib",
    "lgbm_metrics_train.joblib",
    "lgbm_metrics_test.joblib",
    "rf_metrics_train.joblib",
    "rf_metrics_test.joblib",
    "xgb_metrics_train.joblib",
    "xgb_metrics_test.joblib",
]

for metric_file in metric_files:
    path = Path(metric_file)
    if path.exists():
        print(f"\n{metric_file}")
        ml_modeling.read_joblib_metric(path)
    else:
        print(f"Not created yet: {metric_file}")